In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load CSV files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Helper to aggregate numeric telemetry files
def aggregate_numeric(df, name_col, file_type, source_filename):
    g = df.groupby(['cmdb_id', name_col])['value']
    agg = g.agg(
        count='count',
        p95=lambda x: float(np.quantile(x.values, 0.95)) if len(x) > 0 else np.nan,
        p5 = lambda x: float(np.quantile(x.values, 0.05)) if len(x) > 0 else np.nan,
        mean='mean',
        max='max'
    ).reset_index()
    agg = agg.rename(columns={name_col: 'name'})
    agg['source'] = source_filename
    agg['file_type'] = file_type
    # Reorder columns as requested
    agg = agg[['source','file_type','cmdb_id','name','count','p95','p5','mean','max']]
    # Round numeric columns for compactness
    for c in ['p95','p5','mean','max']:
        agg[c] = agg[c].round(6)
    # Save full aggregated table in a variable for later use
    return agg

aggregated_metric = aggregate_numeric(df_metric, 'kpi_name', 'metric', 'metric.csv')
aggregated_trace  = aggregate_numeric(df_trace,  'trace_name', 'trace',  'trace.csv')
aggregated_log    = aggregate_numeric(df_log,    'log_name',   'log',    'log.csv')

# Prepare display tables: top 50 groups sorted by count descending
metric_display = aggregated_metric.sort_values('count', ascending=False).head(50)
trace_display  = aggregated_trace.sort_values('count', ascending=False).head(50)
log_display    = aggregated_log.sort_values('count', ascending=False).head(50)

# Process error_logs.csv per-component aggregates
tz_sh = pytz.timezone('Asia/Shanghai')

def to_shanghai_iso(ts_series):
    # Convert epoch seconds to timezone-aware string in Asia/Shanghai
    return pd.to_datetime(ts_series, unit='s', utc=True).dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

if not df_error.empty:
    grp = df_error.groupby('cmdb_id')
    error_agg = grp.agg(
        count = ('timestamp', 'count'),
        earliest_ts = ('timestamp', 'min'),
        latest_ts = ('timestamp', 'max'),
        # collect up to 3 sample messages
        sample_messages = ('message', lambda s: list(s.astype(str).head(3)))
    ).reset_index()
    # convert timestamps to Asia/Shanghai human-readable
    error_agg['earliest_ts'] = to_shanghai_iso(error_agg['earliest_ts'])
    error_agg['latest_ts'] = to_shanghai_iso(error_agg['latest_ts'])
    # join sample messages into a compact string
    error_agg['sample_messages'] = error_agg['sample_messages'].apply(lambda msgs: " | ".join(msgs))
    # Final columns
    error_logs_agg = error_agg[['cmdb_id','count','earliest_ts','latest_ts','sample_messages']]
else:
    # Empty table
    error_logs_agg = pd.DataFrame(columns=['cmdb_id','count','earliest_ts','latest_ts','sample_messages'])

# Keep full aggregated tables in memory: aggregated_metric, aggregated_trace, aggregated_log, error_logs_agg
# Display compact top results (up to 50 rows each) and the full error_logs per-component table (expected small)
metric_display, trace_display, log_display, error_logs_agg

```
Out[1]:
```
```python
# Prepare a concise plain-English summary string based on the previously computed aggregates.
top_metric_groups = aggregated_metric.sort_values('count', ascending=False).head(3)[['cmdb_id','name','count']].to_records(index=False)
top_trace_groups  = aggregated_trace.sort_values('count', ascending=False).head(3)[['cmdb_id','name','count']].to_records(index=False)
top_log_groups    = aggregated_log.sort_values('count', ascending=False).head(3)[['cmdb_id','name','count']].to_records(False)

metric_total_groups = aggregated_metric.shape[0]
trace_total_groups  = aggregated_trace.shape[0]
log_total_groups    = aggregated_log.shape[0]

metric_max_count = int(aggregated_metric['count'].max()) if metric_total_groups>0 else 0
metric_max_count_num = int((aggregated_metric['count'] == metric_max_count).sum()) if metric_total_groups>0 else 0

trace_max_count = int(aggregated_trace['count'].max()) if trace_total_groups>0 else 0
trace_max_count_num = int((aggregated_trace['count'] == trace_max_count).sum()) if trace_total_groups>0 else 0

log_max_count = int(aggregated_log['count'].max()) if log_total_groups>0 else 0
log_max_count_num = int((aggregated_log['count'] == log_max_count).sum()) if log_total_groups>0 else 0

error_logs_present = not error_logs_agg.empty
error_logs_summary = "no error log records found" if not error_logs_present else (
    f"{error_logs_agg.shape[0]} components with error log records (see variable error_logs_agg)."
)

summary = (
    "Summary of aggregated telemetry (plain English):\n\n"
    f"- Aggregated group counts: metric={metric_total_groups}, trace={trace_total_groups}, log={log_total_groups}.\n\n"
    f"- Metric aggregates: highest group count = {metric_max_count} (occurs in {metric_max_count_num} groups).\n"
    "  Top metric groups (cmdb_id, name, count):\n"
    + "\n".join([f"    - {t[0]} | {t[1][:80]}... | count={t[2]}" for t in top_metric_groups]) + "\n\n"
    f"- Trace aggregates: highest group count = {trace_max_count} (occurs in {trace_max_count_num} groups).\n"
    "  Top trace groups (cmdb_id, name, count):\n"
    + "\n".join([f"    - {t[0]} | {t[1][:80]}... | count={t[2]}" for t in top_trace_groups]) + "\n\n"
    f"- Log aggregates: highest group count = {log_max_count} (occurs in {log_max_count_num} groups).\n"
    "  Top log groups (cmdb_id, name, count):\n"
    + "\n".join([f"    - {t[0]} | {t[1][:80]}... | count={t[2]}" for t in top_log_groups]) + "\n\n"
    f"- Error logs: {error_logs_summary}.\n\n"
    "Notes:\n"
    "1) Full aggregated tables are stored in kernel variables: aggregated_metric, aggregated_trace, aggregated_log, and error_logs_agg.\n"
    "2) The printed top entries above are compact samples; inspect those variables directly for full details (they were computed over the entire available series).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          source file_type   cmdb_id                                               name  count           p95            p5          mean           max
1584  metric.csv    metric  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1585  metric.csv    metric  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1586  metric.csv    metric  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1587  metric.csv    metric  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1588  metric.csv    metric  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
...          ...       ...       ...                                                ...    ...           ...           ...           ...           ...
1452  metric.csv    metric  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  4.016560e+05  4.016560e+05  4.016560e+05  4.016560e+05
1453  metric.csv    metric  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  1.584607e+09  1.584531e+09  1.584569e+09  1.584612e+09
1454  metric.csv    metric  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2.557718e+06  2.556738e+06  2.557229e+06  2.557777e+06
1455  metric.csv    metric  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1445  metric.csv    metric  Tomcat03                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  7.652813e+06  7.651189e+06  7.652001e+06  7.652905e+06

[50 rows x 9 columns],         source file_type   cmdb_id                               name  count         p95          p5        mean         max
260  trace.csv     trace  dockerB2            trace.to_MG02.row_count      1  559.000000  559.000000  559.000000  559.000000
0    trace.csv     trace      IG01      trace.from_root.duration_mean      1    0.055143    0.055143    0.055143    0.055143
1    trace.csv     trace      IG01       trace.from_root.duration_p95      1    0.091300    0.091300    0.091300    0.091300
2    trace.csv     trace      IG01          trace.from_root.row_count      1    7.000000    7.000000    7.000000    7.000000
3    trace.csv     trace      IG01           trace.self.duration_mean      1    0.087221    0.087221    0.087221    0.087221
..         ...       ...       ...                                ...    ...         ...         ...         ...         ...
53   trace.csv     trace      MG01      trace.from_dockerA2.row_count      1   91.000000   91.000000   91.000000   91.000000
54   trace.csv     trace      MG01  trace.from_dockerB1.duration_mean      1    0.012568    0.012568    0.012568    0.012568
55   trace.csv     trace      MG01   trace.from_dockerB1.duration_p95      1    0.014000    0.014000    0.014000    0.014000
56   trace.csv     trace      MG01      trace.from_dockerB1.row_count      1  412.000000  412.000000  412.000000  412.000000
57   trace.csv     trace      MG01  trace.from_dockerB2.duration_mean      1    0.010144    0.010144    0.010144    0.010144

[50 rows x 9 columns],      source file_type   cmdb_id              name  count      p95       p5       mean     max
0   log.csv       log  Tomcat01   log.error_count     30   0.0000   0.0000   0.000000    0.00
2   log.csv       log  Tomcat01     log.row_count     30  54.5500  23.0000  36.633333   56.00
3   log.csv       log  Tomcat02   log.error_count     30   0.0000   0.0000   0.000000    0.00
5   log.csv       log  Tomcat02     log.row_count     30  48.3000  22.2500  35.500000   53.00
6   log.csv       log  Tomcat03   log.error_count     30   0.0000   0.0000   0.000000    0.00
11  log.csv       log  Tomcat04     log.row_count     30  47.0000  26.4500  36.633333   48.00
9   log.csv       log  Tomcat04   log.error_count     30   0.0000   0.0000   0.000000    0.00
8   log.csv       log  Tomcat03     log.row_count     30  53.7500  25.9000  35.766667   61.00
14  log.csv       log  apache02   log.error_count     30   0.0000   0.0000   0.000000    0.00
13  log.csv       log  apache01     log.row_count     30  90.1000  54.1500  70.966667  100.00
15  log.csv       log  apache02     log.row_count     30  93.7500  55.2500  73.300000  107.00
12  log.csv       log  apache01   log.error_count     30   0.0000   0.0000   0.000000    0.00
10  log.csv       log  Tomcat04  log.gc_pause_max      4   0.0785   0.0315   0.055000    0.08
4   log.csv       log  Tomcat02  log.gc_pause_max      2   0.0690   0.0510   0.060000    0.07
7   log.csv       log  Tomcat03  log.gc_pause_max      1   0.0700   0.0700   0.070000    0.07
1   log.csv       log  Tomcat01  log.gc_pause_max      1   0.0800   0.0800   0.080000    0.08, Empty DataFrame
Columns: [cmdb_id, count, earliest_ts, latest_ts, sample_messages]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
import datetime

# Reuse existing variables from kernel: df_metric, df_trace, df_log, aggregated_metric, aggregated_trace, aggregated_log
# If any is missing, raise error to comply with "Do not assume unknown" rule.
_required = ['df_metric','df_trace','df_log','aggregated_metric','aggregated_trace','aggregated_log']
for v in _required:
    if v not in globals():
        raise RuntimeError(f"Required variable not found in kernel: {v}")

# Time window (Asia/Shanghai UTC+8) -> convert to epoch seconds (UTC)
tz_sh = pytz.timezone('Asia/Shanghai')
start_local = tz_sh.localize(datetime.datetime(2021,3,23,6,30,0))
end_local   = tz_sh.localize(datetime.datetime(2021,3,23,7,0,0))
start_ts = int(start_local.astimezone(pytz.UTC).timestamp())
end_ts   = int(end_local.astimezone(pytz.UTC).timestamp())

def to_shanghai_iso(ts_series):
    return pd.to_datetime(ts_series, unit='s', utc=True).dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

def detect_anomalies(orig_df, timestamp_col, cmdb_col, name_col, aggregated_df, file_type_label):
    # Filter to window
    window_df = orig_df[(orig_df[timestamp_col] >= start_ts) & (orig_df[timestamp_col] <= end_ts)].copy()
    # Save full filtered per-point table for later inspection
    # Join global thresholds (p95,p5) by (cmdb_id, name)
    joined = window_df.merge(
        aggregated_df[['cmdb_id','name','p95','p5']],
        left_on=[cmdb_col, name_col],
        right_on=['cmdb_id','name'],
        how='left',
        suffixes=('','_thr')
    )
    # Mark anomalies only where thresholds exist (non-null)
    joined['is_anomaly'] = False
    has_thr = joined['p95'].notna() & joined['p5'].notna()
    joined.loc[has_thr, 'is_anomaly'] = (joined.loc[has_thr,'value'] >= joined.loc[has_thr,'p95']) | (joined.loc[has_thr,'value'] <= joined.loc[has_thr,'p5'])
    # Total groups scanned in window
    groups_scanned = joined[[cmdb_col, name_col]].drop_duplicates().shape[0]
    # Per-group aggregate: need groups that have at least one anomaly
    if joined.empty:
        agg_df = pd.DataFrame(columns=['file_type','cmdb_id','name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
                                       'max_value_in_window','min_value_in_window','threshold_p95','threshold_p5','severity'])
        return joined, agg_df, groups_scanned

    # Compute per-group metrics
    # 1) From anomaly rows: anomaly_count, earliest/latest anomaly ts
    anomaly_rows = joined[joined['is_anomaly']]
    if not anomaly_rows.empty:
        a_grp = anomaly_rows.groupby([cmdb_col, name_col])['timestamp'].agg(
            anomaly_count = 'count',
            earliest_anomaly_ts = 'min',
            latest_anomaly_ts = 'max'
        ).reset_index()
    else:
        a_grp = pd.DataFrame(columns=[cmdb_col, name_col, 'anomaly_count','earliest_anomaly_ts','latest_anomaly_ts'])

    # 2) From all rows in window: max/min value in window and thresholds (take from aggregated join - same per group)
    wm = joined.groupby([cmdb_col, name_col]).agg(
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        threshold_p95 = ('p95','first'),
        threshold_p5  = ('p5','first'),
    ).reset_index()

    # Merge anomaly info into window metrics
    merged = wm.merge(a_grp, on=[cmdb_col, name_col], how='left')

    # Keep only groups with anomaly_count>0
    merged['anomaly_count'] = merged['anomaly_count'].fillna(0).astype(int)
    anomalies_only = merged[merged['anomaly_count']>0].copy()

    if not anomalies_only.empty:
        # Compute severity as max(|max - p95|, |p5 - min|)
        # If thresholds missing, treat diff as 0 (no applicable severity)
        anomalies_only['diff_high'] = (anomalies_only['max_value_in_window'] - anomalies_only['threshold_p95']).abs()
        anomalies_only['diff_low']  = (anomalies_only['threshold_p5'] - anomalies_only['min_value_in_window']).abs()
        anomalies_only['diff_high'] = anomalies_only['diff_high'].fillna(0.0)
        anomalies_only['diff_low']  = anomalies_only['diff_low'].fillna(0.0)
        anomalies_only['severity'] = anomalies_only[['diff_high','diff_low']].max(axis=1)
        # convert timestamps to Asia/Shanghai strings
        anomalies_only['earliest_anomaly_ts'] = to_shanghai_iso(anomalies_only['earliest_anomaly_ts'])
        anomalies_only['latest_anomaly_ts']   = to_shanghai_iso(anomalies_only['latest_anomaly_ts'])
    else:
        anomalies_only['severity'] = pd.Series(dtype=float)

    # Final columns and ordering
    anomalies_only['file_type'] = file_type_label
    anomalies_only = anomalies_only.rename(columns={cmdb_col:'cmdb_id', name_col:'name'})
    final_cols = ['file_type','cmdb_id','name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
                  'max_value_in_window','min_value_in_window','threshold_p95','threshold_p5','severity']
    # Ensure columns exist
    for c in final_cols:
        if c not in anomalies_only.columns:
            anomalies_only[c] = np.nan
    result = anomalies_only[final_cols].copy()
    # Sort by anomaly_count desc then severity desc
    result = result.sort_values(['anomaly_count','severity'], ascending=[False, False]).reset_index(drop=True)
    return joined, result, groups_scanned

# Metric
metric_window_joined, metric_anoms_agg, metric_groups_scanned = detect_anomalies(
    df_metric, 'timestamp', 'cmdb_id', 'kpi_name', aggregated_metric, 'metric'
)
metric_result_top20 = metric_anoms_agg.head(20)
metric_summary = f"metric: scanned {metric_groups_scanned} groups in window, {metric_anoms_agg.shape[0]} groups with anomalies."

# Trace
trace_window_joined, trace_anoms_agg, trace_groups_scanned = detect_anomalies(
    df_trace, 'timestamp', 'cmdb_id', 'trace_name', aggregated_trace, 'trace'
)
trace_result_top20 = trace_anoms_agg.head(20)
trace_summary = f"trace: scanned {trace_groups_scanned} groups in window, {trace_anoms_agg.shape[0]} groups with anomalies."

# Log
log_window_joined, log_anoms_agg, log_groups_scanned = detect_anomalies(
    df_log, 'timestamp', 'cmdb_id', 'log_name', aggregated_log, 'log'
)
log_result_top20 = log_anoms_agg.head(20)
log_summary = f"log: scanned {log_groups_scanned} groups in window, {log_anoms_agg.shape[0]} groups with anomalies."

# Keep full per-point filtered tables and joined thresholds in kernel variables:
# metric_window_joined, trace_window_joined, log_window_joined (already returned)
# And full aggregated per-group anomaly tables stored as metric_anoms_agg, trace_anoms_agg, log_anoms_agg

# Display compact results: top 20 per file and one-line summaries
metric_result_top20, metric_summary, trace_result_top20, trace_summary, log_result_top20, log_summary

```
Out[2]:
```
```python
# Compose a concise plain-English summary based on previously computed results in kernel variables.
# Reuse available variables: metric_result_top20, trace_result_top20, log_result_top20, metric_summary, trace_summary, log_summary

def top_rows_summary(df, n=3):
    rows = df.head(n).to_dict('records')
    lines = []
    for r in rows:
        name = (r.get('name') or '')[:140]
        lines.append(f"{r.get('cmdb_id')} | {name} | count={int(r.get('anomaly_count',0))} | max={r.get('max_value_in_window')} | min={r.get('min_value_in_window')} | severity={float(r.get('severity',0)):.3f}")
    return lines if lines else ["(no top anomalies)"]

metric_top_lines = top_rows_summary(metric_result_top20, 5)
log_top_lines = top_rows_summary(log_result_top20, 5)
trace_top_lines = top_rows_summary(trace_result_top20, 5)

summary = (
    "Incident window summary (2021-03-23 06:30:00 to 2021-03-23 07:00:00, Asia/Shanghai):\n\n"
    f"METRIC: {metric_summary}\n"
    "Top metric anomaly groups (up to 5):\n  - " + "\n  - ".join(metric_top_lines) + "\n\n"
    f"TRACE: {trace_summary}\n"
    "Top trace anomaly groups (up to 5):\n  - " + "\n  - ".join(trace_top_lines) + "\n\n"
    f"LOG: {log_summary}\n"
    "Top log anomaly groups (up to 5):\n  - " + "\n  - ".join(log_top_lines) + "\n\n"
    "Notes:\n"
    "- Trace had no data in the window (0 groups scanned).\n"
    "- Metrics: many groups flagged because global p5/p95 thresholds matched the window values; notable high-severity metric: Mysql02 Mysql-MySQL_3306_Sort Rows (31 anomalies, severity ~18.5).\n"
    "- Logs: all scanned log groups showed anomalies; notable high-severity log row counts on apache01/apache02 and Tomcat row_count entries.\n"
    "- Full per-point filtered & joined tables are kept in kernel variables for inspection: metric_window_joined, metric_anoms_agg, trace_window_joined, trace_anoms_agg, log_window_joined, log_anoms_agg.\n"
    "- One-line summaries are available in variables: metric_summary, trace_summary, log_summary."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   file_type   cmdb_id                                               name  anomaly_count      earliest_anomaly_ts        latest_anomaly_ts  max_value_in_window  min_value_in_window  threshold_p95  threshold_p5  severity
0     metric   Mysql02                         Mysql-MySQL_3306_Sort Rows             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         1.166717e+06         1.166697e+06   1.166717e+06  1.166716e+06      18.5
1     metric      MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         2.000000e+00         0.000000e+00   5.000000e-01  0.000000e+00       1.5
2     metric   Mysql02               Mysql-MySQL_3306_Max trx rows locked             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         1.000000e+00         0.000000e+00   0.000000e+00  0.000000e+00       1.0
3     metric   Mysql02                    Mysql-MySQL_3306_Threads Cached             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         1.700000e+01         1.600000e+01   1.700000e+01  1.700000e+01       1.0
4     metric   Mysql02                  Mysql-MySQL_3306_ThreadsConnected             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         2.100000e+01         2.000000e+01   2.000000e+01  2.000000e+01       1.0
5     metric  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         1.000000e+00         0.000000e+00   0.000000e+00  0.000000e+00       1.0
6     metric  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         1.000000e+00         0.000000e+00   0.000000e+00  0.000000e+00       1.0
7     metric   Mysql02        Mysql-MySQL_3306_Innodb data pending fsyncs             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         1.000000e+00         0.000000e+00   5.000000e-01  0.000000e+00       0.5
8     metric   Mysql02        Mysql-MySQL_3306_Innodb data pending writes             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         1.000000e+00         0.000000e+00   5.000000e-01  0.000000e+00       0.5
9     metric   Mysql02      Mysql-MySQL_3306_Innodb os log pending fsyncs             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         1.000000e+00         0.000000e+00   5.000000e-01  0.000000e+00       0.5
10    metric      MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         4.000000e-01         3.000000e-01   3.000000e-01  3.000000e-01       0.1
11    metric  Tomcat02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         4.000000e-01         3.000000e-01   3.000000e-01  3.000000e-01       0.1
12    metric      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         2.040136e+09         2.040136e+09   2.040136e+09  2.040136e+09       0.0
13    metric      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         5.805683e+07         5.805683e+07   5.805683e+07  5.805683e+07       0.0
14    metric      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         2.040136e+09         2.040136e+09   2.040136e+09  2.040136e+09       0.0
15    metric      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         5.811566e+07         5.811566e+07   5.811566e+07  5.811566e+07       0.0
16    metric      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         2.040136e+09         2.040136e+09   2.040136e+09  2.040136e+09       0.0
17    metric      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         5.894522e+07         5.894522e+07   5.894522e+07  5.894522e+07       0.0
18    metric      MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         0.000000e+00         0.000000e+00   0.000000e+00  0.000000e+00       0.0
19    metric      MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST         2.000000e+00         2.000000e+00   2.000000e+00  2.000000e+00       0.0, 'metric: scanned 1756 groups in window, 1756 groups with anomalies.', Empty DataFrame
Columns: [file_type, cmdb_id, name, anomaly_count, earliest_anomaly_ts, latest_anomaly_ts, max_value_in_window, min_value_in_window, threshold_p95, threshold_p5, severity]
Index: [], 'trace: scanned 0 groups in window, 0 groups with anomalies.',    file_type   cmdb_id              name  anomaly_count      earliest_anomaly_ts        latest_anomaly_ts  max_value_in_window  min_value_in_window  threshold_p95  threshold_p5  severity
0        log  Tomcat01   log.error_count             30  2021-03-23 06:30:00 CST  2021-03-23 06:59:00 CST                 0.00                 0.00         0.0000        0.0000    0.0000
1        log  Tomcat02   log.error_count             30  2021-03-23 06:30:00 CST  2021-03-23 06:59:00 CST                 0.00                 0.00         0.0000        0.0000    0.0000
2        log  Tomcat03   log.error_count             30  2021-03-23 06:30:00 CST  2021-03-23 06:59:00 CST                 0.00                 0.00         0.0000        0.0000    0.0000
3        log  Tomcat04   log.error_count             30  2021-03-23 06:30:00 CST  2021-03-23 06:59:00 CST                 0.00                 0.00         0.0000        0.0000    0.0000
4        log  apache01   log.error_count             30  2021-03-23 06:30:00 CST  2021-03-23 06:59:00 CST                 0.00                 0.00         0.0000        0.0000    0.0000
5        log  apache02   log.error_count             30  2021-03-23 06:30:00 CST  2021-03-23 06:59:00 CST                 0.00                 0.00         0.0000        0.0000    0.0000
6        log  Tomcat04     log.row_count              5  2021-03-23 06:35:00 CST  2021-03-23 06:54:00 CST                48.00                23.00        47.0000       26.4500    3.4500
7        log  Tomcat01     log.row_count              5  2021-03-23 06:30:00 CST  2021-03-23 06:53:00 CST                56.00                21.00        54.5500       23.0000    2.0000
8        log  apache02     log.row_count              4  2021-03-23 06:33:00 CST  2021-03-23 06:56:00 CST               107.00                52.00        93.7500       55.2500   13.2500
9        log  apache01     log.row_count              4  2021-03-23 06:32:00 CST  2021-03-23 06:47:00 CST               100.00                43.00        90.1000       54.1500   11.1500
10       log  Tomcat03     log.row_count              4  2021-03-23 06:33:00 CST  2021-03-23 06:56:00 CST                61.00                24.00        53.7500       25.9000    7.2500
11       log  Tomcat02     log.row_count              4  2021-03-23 06:32:00 CST  2021-03-23 06:47:00 CST                53.00                20.00        48.3000       22.2500    4.7000
12       log  Tomcat04  log.gc_pause_max              2  2021-03-23 06:38:00 CST  2021-03-23 06:48:00 CST                 0.08                 0.03         0.0785        0.0315    0.0015
13       log  Tomcat02  log.gc_pause_max              2  2021-03-23 06:42:00 CST  2021-03-23 06:59:00 CST                 0.07                 0.05         0.0690        0.0510    0.0010
14       log  Tomcat01  log.gc_pause_max              1  2021-03-23 06:43:00 CST  2021-03-23 06:43:00 CST                 0.08                 0.08         0.0800        0.0800    0.0000
15       log  Tomcat03  log.gc_pause_max              1  2021-03-23 06:45:00 CST  2021-03-23 06:45:00 CST                 0.07                 0.07         0.0700        0.0700    0.0000, 'log: scanned 16 groups in window, 16 groups with anomalies.')```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_window_joined, metric_anoms_agg, aggregated_metric
_required = ['metric_window_joined','metric_anoms_agg','aggregated_metric']
for v in _required:
    if v not in globals():
        raise RuntimeError(f"Required variable not found in kernel: {v}")

tz_sh = pytz.timezone('Asia/Shanghai')

# 1) Select top 10 metric groups by anomaly_count then severity, ensure Mysql02 groups included
ranked = metric_anoms_agg.sort_values(['anomaly_count','severity'], ascending=[False, False]).reset_index(drop=True)
# desired key columns in metric_anoms_agg: cmdb_id, name
top_candidates = ranked[['cmdb_id','name','anomaly_count','severity']].copy()

# Ensure Mysql02 groups included if they exist with anomalies
mysql_groups = top_candidates[top_candidates['cmdb_id']=='Mysql02'][['cmdb_id','name','anomaly_count','severity']]

# Build final selected list: take top rows then ensure inclusion of Mysql02 groups, keep uniqueness
selected = []
for _, row in top_candidates.iterrows():
    key = (row['cmdb_id'], row['name'])
    if key not in selected:
        selected.append(key)
    if len(selected) >= 10:
        break

# Add Mysql02 groups if not present (while keeping length <= 10)
for _, row in mysql_groups.iterrows():
    key = (row['cmdb_id'], row['name'])
    if key not in selected:
        if len(selected) < 10:
            selected.append(key)
        else:
            # replace last if we must ensure at least one Mysql02 included
            # but only do this if there is at least one Mysql02 in metric_anoms_agg
            selected[-1] = key
            break

# Trim to at most 10
selected = selected[:10]

# 2) For each selected group produce consecutive anomaly intervals from metric_window_joined
# Ensure the per-point joined table has expected columns: timestamp, cmdb_id, kpi_name, value, p95, p5, is_anomaly
mw = metric_window_joined  # reuse
expected_cols = ['timestamp','cmdb_id','kpi_name','value','p95','p5','is_anomaly']
for c in ['timestamp','cmdb_id','kpi_name','value','p95','p5','is_anomaly']:
    if c not in mw.columns:
        raise RuntimeError(f"Expected column '{c}' not found in metric_window_joined")

intervals = []
for cmdb_id, kpi_name in selected:
    grp = mw[(mw['cmdb_id']==cmdb_id) & (mw['kpi_name']==kpi_name)].copy()
    if grp.empty:
        continue
    # anomaly rows only
    anom = grp[grp['is_anomaly']].sort_values('timestamp')
    if anom.empty:
        continue
    # compute consecutive groups where timestamp difference == 60s
    anom = anom.assign(ts=anom['timestamp'].astype(int))
    anom['ts_diff'] = anom['ts'].diff().fillna(0).astype(int)
    # new interval when diff != 60
    anom['interval_id'] = (anom['ts_diff'] != 60).cumsum()
    # aggregate per interval
    agg = anom.groupby('interval_id').agg(
        interval_start_ts = ('ts','min'),
        interval_end_ts = ('ts','max'),
        interval_anomaly_count = ('ts','count'),
        interval_max_value = ('value','max'),
        interval_min_value = ('value','min'),
        threshold_p95 = ('p95','first'),
        threshold_p5  = ('p5','first')
    ).reset_index(drop=True)
    # compute interval_length_minutes
    agg['interval_length_minutes'] = ((agg['interval_end_ts'] - agg['interval_start_ts'])/60).astype(int) + 1
    # compute breach_ratio per instruction
    def compute_breach_ratio(row):
        p95 = row['threshold_p95']
        maxv = row['interval_max_value']
        if pd.isna(p95):
            return np.nan
        if float(p95) == 0.0:
            return float(abs(maxv - p95))
        else:
            return float((maxv - p95) / max(float(p95), 1e-9))
    agg['breach_ratio'] = agg.apply(compute_breach_ratio, axis=1)
    # severity_label
    def severity_label(row):
        br = row['breach_ratio'] if not pd.isna(row['breach_ratio']) else 0.0
        p5 = row['threshold_p5']
        minv = row['interval_min_value']
        # check low-side breach condition (interval_min_value <= threshold_p5 - 0.5*abs(threshold_p5))
        low_breach = False
        if not pd.isna(p5) and not pd.isna(minv):
            low_breach = (minv <= (p5 - 0.5*abs(p5)))
        if br >= 0.5 or low_breach:
            return 'high'
        if br >= 0.1:
            return 'medium'
        return 'low'
    agg['severity_label'] = agg.apply(severity_label, axis=1)
    # attach identifiers
    agg['cmdb_id'] = cmdb_id
    agg['name'] = kpi_name
    # reorder columns as required
    agg = agg[['cmdb_id','name','interval_start_ts','interval_end_ts','interval_length_minutes',
               'interval_anomaly_count','interval_max_value','interval_min_value',
               'threshold_p95','threshold_p5','breach_ratio','severity_label']]
    # append all intervals for this group
    for _, row in agg.iterrows():
        intervals.append(row.to_dict())

# Create DataFrame of all found intervals
if intervals:
    metric_faults_agg = pd.DataFrame(intervals)
else:
    metric_faults_agg = pd.DataFrame(columns=['cmdb_id','name','interval_start_ts','interval_end_ts','interval_length_minutes',
                                              'interval_anomaly_count','interval_max_value','interval_min_value',
                                              'threshold_p95','threshold_p5','breach_ratio','severity_label'])

# 4) Return only the top interval per group (largest by interval_anomaly_count) for up to 10 groups
if not metric_faults_agg.empty:
    # pick top interval per group
    metric_faults_agg['rank_within_group'] = metric_faults_agg.groupby(['cmdb_id','name'])['interval_anomaly_count'] \
                                               .rank(method='first', ascending=False)
    top_per_group = metric_faults_agg[metric_faults_agg['rank_within_group']==1].copy()
    # sort by interval_anomaly_count desc then breach_ratio desc
    top_per_group = top_per_group.sort_values(['interval_anomaly_count','breach_ratio'], ascending=[False, False]).reset_index(drop=True)
    metric_faults_agg_top10 = top_per_group.head(10).copy()
    # derive earliest interval_start_ts among returned intervals (as Asia/Shanghai string)
    earliest_ts_epoch = int(metric_faults_agg_top10['interval_start_ts'].min())
    earliest_ts_sh = pd.to_datetime(earliest_ts_epoch, unit='s', utc=True).tz_convert(tz_sh).strftime('%Y-%m-%d %H:%M:%S %Z')
    summary = f"scanned_groups={len(selected)}, returned_intervals={metric_faults_agg_top10.shape[0]}, earliest_interval_start={earliest_ts_sh}"
else:
    metric_faults_agg_top10 = metric_faults_agg.copy()
    summary = f"scanned_groups={len(selected)}, returned_intervals=0, earliest_interval_start=None"

# Convert interval_start_ts and interval_end_ts to Asia/Shanghai human-readable strings for compact display
def to_shanghai_str(epoch):
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz_sh).strftime('%Y-%m-%d %H:%M:%S %Z')

if not metric_faults_agg_top10.empty:
    metric_faults_agg_top10_display = metric_faults_agg_top10.copy()
    metric_faults_agg_top10_display['interval_start_ts'] = metric_faults_agg_top10_display['interval_start_ts'].apply(to_shanghai_str)
    metric_faults_agg_top10_display['interval_end_ts']   = metric_faults_agg_top10_display['interval_end_ts'].apply(to_shanghai_str)
    # Keep displayed columns compact and ordered
    metric_faults_agg_top10_display = metric_faults_agg_top10_display[['cmdb_id','name','interval_start_ts','interval_end_ts',
                                                                       'interval_length_minutes','interval_anomaly_count',
                                                                       'interval_max_value','interval_min_value',
                                                                       'threshold_p95','threshold_p5','breach_ratio','severity_label']].reset_index(drop=True)
else:
    metric_faults_agg_top10_display = metric_faults_agg_top10.copy()

# Save metric_window_joined (already present) and metric_faults_agg (full intervals) in kernel for later steps
# metric_window_joined is reused; metric_faults_agg is created above

# Final compact outputs
metric_faults_agg_top10_display, summary

```
Out[3]:
```
```python
# Compose a plain-English summary string of the fault-interval results using existing kernel variables.
# Reuse metric_faults_agg_top10_display and summary produced earlier.

# Ensure variables exist
_required = ['metric_faults_agg_top10_display','summary']
for v in _required:
    if v not in globals():
        raise RuntimeError(f"Required variable not found in kernel: {v}")

df = metric_faults_agg_top10_display.copy()

# Build compact lines for up to 10 intervals
lines = []
for _, r in df.head(10).iterrows():
    lines.append(
        f"{r['cmdb_id']} | {str(r['name'])[:80]} | length={int(r['interval_length_minutes'])}m | "
        f"anomalies={int(r['interval_anomaly_count'])} | max={float(r['interval_max_value']):.3g} | "
        f"min={float(r['interval_min_value']):.3g} | p95={float(r['threshold_p95']):.3g} | "
        f"p5={float(r['threshold_p5']):.3g} | breach_ratio={float(r['breach_ratio']):.3g} | {r['severity_label']}"
    )

summary_text = (
    "Fault interval summary (2021-03-23 06:30:00 to 07:00:00 Asia/Shanghai):\n\n"
    f"- Groups scanned/returned (this step): {summary}\n"
    "- Top intervals (one per selected group, up to 10):\n  - " + "\n  - ".join(lines) + "\n\n"
    "- Notes:\n"
    "  * All returned intervals start at 2021-03-23 06:30:00 and span the full 31-minute window (continuous anomalies every minute).\n"
    "  * High-severity breaches include:\n"
    "    - MG01: OSLinux NETWORK TCP-FIN-WAIT (breach_ratio ~3.0) — strong high-side spike.\n"
    "    - Mysql02: Mysql-MySQL_3306_Max trx rows locked and related InnoDB pending fsync/write metrics (breach_ratio ~1.0) — sustained anomalies across the window.\n"
    "    - Tomcat01/Tomcat04: thread-current metrics showing sustained 0->1 anomalies (breach_ratio ~1.0) — sustained state change.\n"
    "    - Mysql02: Mysql-MySQL_3306_Aborted Clients shows high p95 and is marked high severity.\n"
    "  * Several other Mysql02 groups show anomalies but with small relative breach ratios (classified low/medium).\n"
    "\nFull per-point joined table and the full intervals table are saved in kernel variables for inspection:\n"
    "  - metric_window_joined (per-minute points in window with thresholds and anomaly flag)\n"
    "  - metric_faults_agg (all detected intervals)\n"
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                               name        interval_start_ts          interval_end_ts  interval_length_minutes  interval_anomaly_count  interval_max_value  interval_min_value  threshold_p95  threshold_p5  breach_ratio severity_label
0      MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31                 2.0                 0.0            0.5           0.0          3.00           high
1   Mysql02               Mysql-MySQL_3306_Max trx rows locked  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31                 1.0                 0.0            0.0           0.0          1.00           high
2  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31                 1.0                 0.0            0.0           0.0          1.00           high
3  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31                 1.0                 0.0            0.0           0.0          1.00           high
4   Mysql02        Mysql-MySQL_3306_Innodb data pending fsyncs  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31                 1.0                 0.0            0.5           0.0          1.00           high
5   Mysql02        Mysql-MySQL_3306_Innodb data pending writes  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31                 1.0                 0.0            0.5           0.0          1.00           high
6   Mysql02                  Mysql-MySQL_3306_ThreadsConnected  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31                21.0                20.0           20.0          20.0          0.05            low
7   Mysql02                         Mysql-MySQL_3306_Sort Rows  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31           1166717.0           1166697.0      1166717.0     1166715.5          0.00            low
8   Mysql02                    Mysql-MySQL_3306_Threads Cached  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31                17.0                16.0           17.0          17.0          0.00            low
9   Mysql02                   Mysql-MySQL_3306_Aborted Clients  2021-03-23 06:30:00 CST  2021-03-23 07:00:00 CST                       31                      31              5970.0                 0.0         5970.0           0.0          0.00           high, 'scanned_groups=10, returned_intervals=10, earliest_interval_start=2021-03-23 06:30:00 CST')```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables
_required = ['metric_window_joined','log_window_joined','trace_window_joined','aggregated_metric','aggregated_log','aggregated_trace']
for v in _required:
    if v not in globals():
        raise RuntimeError(f"Required variable not found in kernel: {v}")

tz_sh = pytz.timezone('Asia/Shanghai')

def to_shanghai_iso(ts_series):
    return pd.to_datetime(ts_series, unit='s', utc=True).dt.tz_convert(tz_sh).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Helper to extract up to N anomaly rows per group matching conditions from a per-point joined table
def extract_top_anomalies_per_group(joined_df, cmdb_filter, name_filter_func, name_col, max_per_group=3):
    # Ensure joined_df has expected columns: timestamp, cmdb_id, <name_col>, value, p95, p5, is_anomaly
    for c in ['timestamp','cmdb_id', name_col, 'value']:
        if c not in joined_df.columns:
            raise RuntimeError(f"Expected column '{c}' not found in joined per-point table")
    # If thresholds or is_anomaly missing, try to join thresholds from aggregated table
    # But per instructions we use kernel-saved joined tables; assume p95/p5/is_anomaly present, else attempt to fill from aggregated_metric/log
    if 'p95' not in joined_df.columns or 'p5' not in joined_df.columns:
        # attempt to join from aggregated_metric or aggregated_log depending on name_col
        if name_col == 'kpi_name':
            agg = aggregated_metric[['cmdb_id','name','p95','p5']].rename(columns={'name':'kpi_name'})
        elif name_col == 'log_name':
            agg = aggregated_log[['cmdb_id','name','p95','p5']].rename(columns={'name':'log_name'})
        elif name_col == 'trace_name':
            agg = aggregated_trace[['cmdb_id','name','p95','p5']].rename(columns={'name':'trace_name'})
        else:
            agg = None
        if agg is not None:
            joined_df = joined_df.merge(agg, on=['cmdb_id', name_col], how='left')

    if 'is_anomaly' not in joined_df.columns:
        # compute is_anomaly using p95/p5 if available
        if 'p95' in joined_df.columns and 'p5' in joined_df.columns:
            has_thr = joined_df['p95'].notna() & joined_df['p5'].notna()
            joined_df['is_anomaly'] = False
            joined_df.loc[has_thr, 'is_anomaly'] = (joined_df.loc[has_thr,'value'] >= joined_df.loc[has_thr,'p95']) | (joined_df.loc[has_thr,'value'] <= joined_df.loc[has_thr,'p5'])
        else:
            joined_df['is_anomaly'] = False

    # apply cmdb_filter and name_filter_func
    sel = joined_df['cmdb_id'].astype(str) == str(cmdb_filter)
    name_mask = joined_df[name_col].astype(str).apply(name_filter_func)
    df_sel = joined_df[sel & name_mask].copy()
    if df_sel.empty:
        return pd.DataFrame(columns=['cmdb_id','name','timestamp','value','threshold_p95','threshold_p5','anomaly_flag'])
    # focus on anomaly rows only (as requested: return anomaly points)
    anom = df_sel[df_sel['is_anomaly']].sort_values('timestamp')
    if anom.empty:
        return pd.DataFrame(columns=['cmdb_id','name','timestamp','value','threshold_p95','threshold_p5','anomaly_flag'])
    # take up to max_per_group rows
    topn = anom.groupby(['cmdb_id', name_col], sort=False).head(max_per_group).copy()
    # normalize columns
    out = pd.DataFrame({
        'cmdb_id': topn['cmdb_id'],
        'name': topn[name_col],
        'timestamp': to_shanghai_iso(topn['timestamp']),
        'value': topn['value'],
        'threshold_p95': topn.get('p95', np.nan),
        'threshold_p5': topn.get('p5', np.nan),
        'anomaly_flag': topn['is_anomaly'].astype(bool)
    }).reset_index(drop=True)
    return out

# Define metric target selectors (substrings); name must contain substring(s)
metric_targets = [
    ('MG01', lambda s: 'NETWORK_TCP-FIN-WAIT'.lower() in s.lower()),
    ('Mysql02', lambda s: 'Max trx rows locked'.lower() in s.lower()),
    ('Mysql02', lambda s: 'Sort Rows'.lower() in s.lower()),
    ('Mysql02', lambda s: 'Innodb data pending fsyncs'.lower() in s.lower()),
    ('Tomcat01', lambda s: ('Tomcat-Threads'.lower() in s.lower()) and ('CurrentThr'.lower() in s.lower())),
    ('Tomcat04', lambda s: ('Tomcat-Threads'.lower() in s.lower()) and ('CurrentThr'.lower() in s.lower())),
]

# Extract up to 3 anomaly points per metric group
metric_rows_list = []
for cmdb, name_filter in metric_targets:
    df_pts = extract_top_anomalies_per_group(metric_window_joined, cmdb, name_filter, name_col='kpi_name', max_per_group=3)
    if not df_pts.empty:
        metric_rows_list.append(df_pts)

if metric_rows_list:
    metric_inspect_df = pd.concat(metric_rows_list, ignore_index=True)
else:
    metric_inspect_df = pd.DataFrame(columns=['cmdb_id','name','timestamp','value','threshold_p95','threshold_p5','anomaly_flag'])

# Now log targets
log_targets = [
    ('apache01', lambda s: 'log.row_count'.lower() in s.lower()),
    ('apache02', lambda s: 'log.row_count'.lower() in s.lower()),
    ('Tomcat01', lambda s: 'log.row_count'.lower() in s.lower()),
    ('Tomcat04', lambda s: 'log.row_count'.lower() in s.lower()),
]

log_rows_list = []
for cmdb, name_filter in log_targets:
    df_pts = extract_top_anomalies_per_group(log_window_joined, cmdb, name_filter, name_col='log_name', max_per_group=3)
    if not df_pts.empty:
        log_rows_list.append(df_pts)

if log_rows_list:
    log_inspect_df = pd.concat(log_rows_list, ignore_index=True)
else:
    log_inspect_df = pd.DataFrame(columns=['cmdb_id','name','timestamp','value','threshold_p95','threshold_p5','anomaly_flag'])

# Trace: return up to 20 rows within window (as-is). Show compact columns timestamp, cmdb_id, trace_name, value
trace_df = trace_window_joined.copy()
if trace_df.empty:
    trace_inspect_df = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value'])
else:
    # ensure columns present
    for c in ['timestamp','cmdb_id','trace_name','value']:
        if c not in trace_df.columns:
            raise RuntimeError(f"Expected column '{c}' in trace_window_joined")
    trace_inspect_df = trace_df[['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp').head(20).copy()
    trace_inspect_df['timestamp'] = to_shanghai_iso(trace_inspect_df['timestamp'])

# Combine metric and log returned rows to compute earliest anomaly timestamp across returned rows
combined_returned = pd.concat([metric_inspect_df[['cmdb_id','name','timestamp','value','threshold_p95','threshold_p5','anomaly_flag']].assign(source='metric'),
                              log_inspect_df[['cmdb_id','name','timestamp','value','threshold_p95','threshold_p5','anomaly_flag']].assign(source='log')],
                             ignore_index=True)

# Convert timestamp strings back to epoch to find earliest anomaly (they are in Asia/Shanghai strings)
if not combined_returned.empty:
    # parse timestamps back to UTC then find min
    parsed = pd.to_datetime(combined_returned['timestamp']).dt.tz_localize(tz_sh.zone, ambiguous='NaT', nonexistent='NaT')
    # above may produce NaT; better parse explicitly as tz-aware using UTC then convert
    try:
        parsed = pd.to_datetime(combined_returned['timestamp'], utc=True).dt.tz_convert(tz_sh)
    except Exception:
        # fallback: parse without UTC then localize
        parsed = pd.to_datetime(combined_returned['timestamp']).dt.tz_localize(tz_sh)
    earliest_idx = parsed.idxmin()
    earliest_anomaly_sh = combined_returned.loc[earliest_idx, 'timestamp']
else:
    earliest_anomaly_sh = None

# Enforce overall row limit of 50: we already limited per-group to 3 and trace to 20, so combine size likely <=50.
# But to be safe, trim metric_inspect_df and log_inspect_df if needed, prioritizing keeping up to 50 total.
total_rows = len(metric_inspect_df) + len(log_inspect_df) + len(trace_inspect_df)
if total_rows > 50:
    # calculate how many to keep from trace first then logs then metrics (trace less critical). We'll trim trace then logs.
    allowed = 50
    keep_trace = min(len(trace_inspect_df), allowed)
    allowed -= keep_trace
    keep_log = min(len(log_inspect_df), allowed)
    allowed -= keep_log
    keep_metric = min(len(metric_inspect_df), allowed)
    metric_inspect_df = metric_inspect_df.head(keep_metric)
    log_inspect_df = log_inspect_df.head(keep_log)
    trace_inspect_df = trace_inspect_df.head(keep_trace)

# Final compact outputs (variables to display)
metric_inspect_df, log_inspect_df, trace_inspect_df, earliest_anomaly_sh

```
Out[4]:
```
```python
# Compose a plain-English summary based on kernel-saved inspection tables.
# Reuse existing kernel variables: metric_inspect_df, log_inspect_df, trace_inspect_df, earliest_anomaly_sh

_required = ['metric_inspect_df','log_inspect_df','trace_inspect_df','earliest_anomaly_sh']
for v in _required:
    if v not in globals():
        raise RuntimeError(f"Required variable not found in kernel: {v}")

# Helper to build compact lines for first anomaly per group
def first_anomaly_lines(df, source_label):
    lines = []
    if df.empty:
        return lines
    # group by cmdb_id + name and take the first anomaly row
    grp = df.groupby(['cmdb_id','name'], sort=False)
    for (cmdb, name), g in grp:
        row = g.iloc[0]
        ts = row['timestamp']
        val = row['value']
        p95 = row.get('threshold_p95', None)
        p5  = row.get('threshold_p5', None)
        lines.append(f"{source_label} | {cmdb} | {str(name)[:80]} | first_anomaly={ts} | value={val} | p95={p95} | p5={p5}")
    return lines

metric_lines = first_anomaly_lines(metric_inspect_df, 'METRIC')
log_lines = first_anomaly_lines(log_inspect_df, 'LOG')
trace_lines = []
if not trace_inspect_df.empty:
    # show up to 20 trace rows (as requested previously)
    for _, r in trace_inspect_df.head(20).iterrows():
        trace_lines.append(f"TRACE | {r['cmdb_id']} | {r['trace_name'][:80]} | ts={r['timestamp']} | value={r['value']}")

# Build summary string
summary = (
    "Focused inspection summary (first anomaly points per target group):\n\n"
    f"Earliest anomaly across returned rows: {earliest_anomaly_sh}\n\n"
    "Metric groups (first anomaly sample per group):\n"
    + ("\n".join(metric_lines) if metric_lines else "  (no metric anomalies returned)\n")
    + "\n\nLog groups (first anomaly sample per group):\n"
    + ("\n".join(log_lines) if log_lines else "  (no log anomalies returned)\n")
    + "\n\nTrace records in window (up to 20 rows):\n"
    + ("\n".join(trace_lines) if trace_lines else "  (no trace records in the window)\n")
    + "\n\nNotes:\n"
    "- All inspected metric groups show anomalies starting at 2021-03-23 06:30:00 CST.\n"
    "- MG01 NETWORK_TCP-FIN-WAIT: first anomalies at 06:30 with value 0.0 vs p95=0.5 (low-side).\n"
    "- Mysql02 groups (Max trx rows locked, Sort Rows, Innodb pending fsyncs, etc.) show sustained anomalies across the window starting 06:30; Sort Rows shows very large values near p95.\n"
    "- Tomcat thread metrics show state changes (0 ↔ 1 / 100) starting at 06:30.\n"
    "- Log row_count anomalies: apache01/apache02 and Tomcat nodes have first anomaly samples between 06:30–06:35 (see lines above) indicating elevated or boundary log volumes.\n"
    "- Trace data: no trace records found in the window.\n\n"
    "Kernel variables with full per-point joined tables (not printed here) are available:\n"
    "  - metric_window_joined, log_window_joined, trace_window_joined\n\n"
    "Use these variables for any further detailed point-by-point inspection."
)

# Display the summary string as the final output
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                               name                timestamp  value  threshold_p95  threshold_p5  anomaly_flag
0       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-23 06:30:00 CST    0.0            0.5           0.0          True
1       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-23 06:31:00 CST    0.0            0.5           0.0          True
2       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  2021-03-23 06:32:00 CST    0.0            0.5           0.0          True
3    Mysql02               Mysql-MySQL_3306_Max trx rows locked  2021-03-23 06:30:00 CST    0.0            0.0           0.0          True
4    Mysql02               Mysql-MySQL_3306_Max trx rows locked  2021-03-23 06:31:00 CST    0.0            0.0           0.0          True
..       ...                                                ...                      ...    ...            ...           ...           ...
19  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  2021-03-23 06:30:00 CST    0.0            0.0           0.0          True
20  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  2021-03-23 06:31:00 CST  100.0          100.0         100.0          True
21  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  2021-03-23 06:31:00 CST    0.0            0.0           0.0          True
22  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  2021-03-23 06:32:00 CST  100.0          100.0         100.0          True
23  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  2021-03-23 06:32:00 CST    0.0            0.0           0.0          True

[24 rows x 7 columns],      cmdb_id           name                timestamp  value  threshold_p95  threshold_p5  anomaly_flag
0   apache01  log.row_count  2021-03-23 06:32:00 CST   51.0          90.10         54.15          True
1   apache01  log.row_count  2021-03-23 06:33:00 CST   43.0          90.10         54.15          True
2   apache01  log.row_count  2021-03-23 06:42:00 CST   91.0          90.10         54.15          True
3   apache02  log.row_count  2021-03-23 06:33:00 CST   52.0          93.75         55.25          True
4   apache02  log.row_count  2021-03-23 06:42:00 CST  107.0          93.75         55.25          True
5   apache02  log.row_count  2021-03-23 06:46:00 CST   96.0          93.75         55.25          True
6   Tomcat01  log.row_count  2021-03-23 06:30:00 CST   23.0          54.55         23.00          True
7   Tomcat01  log.row_count  2021-03-23 06:33:00 CST   23.0          54.55         23.00          True
8   Tomcat01  log.row_count  2021-03-23 06:41:00 CST   56.0          54.55         23.00          True
9   Tomcat04  log.row_count  2021-03-23 06:35:00 CST   26.0          47.00         26.45          True
10  Tomcat04  log.row_count  2021-03-23 06:39:00 CST   48.0          47.00         26.45          True
11  Tomcat04  log.row_count  2021-03-23 06:43:00 CST   47.0          47.00         26.45          True, Empty DataFrame
Columns: [timestamp, cmdb_id, trace_name, value]
Index: [], '2021-03-23 06:30:00 CST')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```